# 07 · Your data

> **Old question:** Could this work on *my* texts?
> **New tool:** the same three steps as notebook 01: load a CSV, write a codebook, run.
> **What you'll see:** your own texts turned into a coded table, saved as a CSV you can open in Excel, Stata or R.

## Step 1 · Load your file

Your CSV needs two columns: `id` and `text`. Put it in the `data/` folder and change the file name below. To start, we use `my_data_example.csv`: ten short, **synthetic** sentences about firms and work.

**Before you upload anything:** with a live API, your texts are sent to the provider. Do not use confidential or personal data unless your data agreement allows it. In dry-run mode nothing leaves your laptop.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from typing import Literal
import pandas as pd
from pydantic import BaseModel, Field
from workshop import *

status()
my_data = pd.read_csv("../data/my_data_example.csv")   # <- change the file name
print(len(my_data), "rows"); my_data.head()

## Step 2 · Write your codebook

Edit the class below. A few tips:

- one field per variable, with a **description** written like a coding instruction;
- use `Literal["a", "b", "c"]` for closed categories, `bool` for yes/no, `int | None` for numbers that may be missing;
- keep an `evidence_quote` so you can check the answers with `quote_check`.

In [ ]:
class MyCode(BaseModel):
    topic: Literal["work & labour", "trade & policy", "innovation & R&D", "finance", "other"]
    actor: str = Field(description="The main firm, group or institution acting in the text, in 1-4 words")
    tone: Literal["positive", "negative", "mixed", "neutral"] = Field(description="How the outcome is described")
    mentions_technology: bool = Field(description="True if a technology plays a role")
    evidence_quote: str = Field(description="A short phrase copied word for word from the text that supports the topic")

INSTRUCTIONS = "You are coding short research texts. Fill in every field using only the text."

## Step 3 · Test on 3 rows, then run everything

Always test on a few rows first, then read the answers before spending calls on the full file.

In [ ]:
test = extract_many(my_data.head(3), "text", MyCode, INSTRUCTIONS)
test

In [ ]:
coded = extract_many(my_data, "text", MyCode, INSTRUCTIONS)
coded["quote_ok"] = [quote_check(q, t) for q, t in zip(coded["evidence_quote"], coded["text"])]
print(f"failed rows: {coded.error.notna().sum()} · quotes not found in text: {(~coded.quote_ok).sum()}")
coded

In [ ]:
pathlib.Path("../outputs").mkdir(exist_ok=True)
coded.to_csv("../outputs/my_data_coded.csv", index=False, encoding="utf-8-sig")  # utf-8-sig opens cleanly in Excel
print("saved outputs/my_data_coded.csv")

## What did this cost?

Every answer comes with a count of **tokens** (pieces of words; about 4 characters of English each): tokens **in** (your instructions plus the text) and tokens **out** (the answer, including the model's hidden "thinking"). Providers charge per million tokens; input the provider has seen recently (the same instructions, sent again) is **cached** and billed at a tenth of the price. Every cell that got answers printed its own cost line; here is the same arithmetic for the whole notebook. Saved answers keep the counts from the original live call, so this is what the run costs with a live key. Run it after Step 3 to estimate the cost of your whole dataset.

In [ ]:
PRICE_IN, PRICE_CACHED, PRICE_OUT = CFG["price_in"], CFG["price_cached"], CFG["price_out"]   # per million tokens
CUR = CFG["currency"]                                     # ¥1.00 / ¥0.10 / ¥4.00 for MiniMax M3.1 Flash (official, Nov 2026)
u = usage_table()
fresh_in = u.tokens_in.sum() - u.cached_in.sum()          # input the provider had not seen before
cost = (fresh_in * PRICE_IN + u.cached_in.sum() * PRICE_CACHED + u.tokens_out.sum() * PRICE_OUT) / 1e6
print(f"{len(u)} answers: {u.tokens_in.sum():,} tokens in, {u.tokens_out.sum():,} tokens out ({u.thinking.sum():,} of them thinking)")
print(f"cost: {CUR}{cost:.4f}   per answer: {CUR}{cost / max(len(u), 1):.5f}   for 10,000 texts: about {CUR}{cost / max(len(u), 1) * 10_000:.2f}")

## Try it yourself

- Add a variable to `MyCode` and re-run Step 3.
- Run the coding twice with `temperature=0.7` (as in notebook 02) and compare. Which of your variables are least stable?
- Hand-code 20 rows yourself and compare, with the agreement measures from notebook 02.

In [ ]:
# Your turn: a second run at a higher temperature, compared with the first
again = extract_many(my_data, "text", MyCode, INSTRUCTIONS, temperature=0.7)
print(f"topic agreement between runs: {(again.topic == coded.topic).mean():.0%}")